# ROGII: Particle Seed Independence Audit

A different random-seed label does not automatically mean an independent stochastic replication.

This notebook demonstrates a subtle ensemble bookkeeping failure in a GR-guided particle tracker. If an eight-seed ensemble uses the consecutive range `[base, base + 7]`, changing `base` to `base + 1` preserves seven of eight paths. Agreement between those two runs is therefore expected and must not be reported as independent confirmation.

We compare three deterministic runs on the same training-well sample:

- original base `20260717`;
- misleading check `20260718`, which overlaps 7/8 seeds;
- disjoint check `20270717`, which overlaps 0/8 seeds.

Every run sees only the supplied visible `TVT_input` prefix before predicting the true suffix. This is a research audit: it creates no `submission.csv`, uses no public prediction artifact, and spends no competition submission.

In [ ]:
from pathlib import Path
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
TRAIN_DIR = DATA_ROOT / 'train'

SAMPLE_SIZE = 30
SAMPLE_SEED = 20260717
N_PARTICLES = 300
N_SEEDS = 8
SEED_TEMPERATURE = 20.0
PARTICLE_WEIGHT = 0.625
BASES = {
    'original': 20260717,
    'overlap_plus_one': 20260718,
    'disjoint': 20270717,
}

all_paths = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
paths = sorted(np.random.default_rng(SAMPLE_SEED).choice(all_paths, SAMPLE_SIZE, replace=False))
print('Data root:', DATA_ROOT)
print('Deterministic sample:', len(paths), 'of', len(all_paths), 'wells')

## Audit the seed sets before running the model

The well-specific offset below keeps streams separate between wells. It does not change the intersection size between ensemble ranges. The important object is the complete seed set, not the base label.

In [ ]:
example_well_offset = 1000
seed_sets = {
    name: set(range(base + example_well_offset, base + example_well_offset + N_SEEDS))
    for name, base in BASES.items()
}
for name, values in seed_sets.items():
    print(f'{name:18s}', sorted(values))
print('original vs +1 intersection:', len(seed_sets['original'] & seed_sets['overlap_plus_one']), '/', N_SEEDS)
print('original vs disjoint intersection:', len(seed_sets['original'] & seed_sets['disjoint']), '/', N_SEEDS)
assert len(seed_sets['original'] & seed_sets['overlap_plus_one']) == 7
assert len(seed_sets['original'] & seed_sets['disjoint']) == 0

## Same model, three seed sets

The particle state is `U = TVT + Z` plus a slow `dU/dMD` rate. Prefix-only affine calibration maps typewell GR to the horizontal-well GR scale. Suffix GR supplies the particle likelihood, systematic resampling controls degeneracy, and tempered sequence evidence combines eight paths. A fixed 62.5% posterior / 37.5% anchor blend limits wrong-branch damage.

Only the random seed range changes between the three runs.

In [ ]:
def split_boundary(frame):
    observed = frame['TVT_input'].notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] < 2 or observed[missing[0]:].any():
        raise ValueError('Expected a visible prefix and one contiguous missing suffix')
    return int(missing[0])

def calibrate_gr(expected, observed):
    mask = np.isfinite(expected) & np.isfinite(observed)
    x, y = expected[mask], observed[mask]
    if len(x) < 20:
        return 1.0, 0.0, 30.0
    keep = np.ones(len(x), dtype=bool)
    gain, offset = 1.0, 0.0
    for _ in range(3):
        design = np.column_stack((x[keep], np.ones(int(keep.sum()))))
        gain, offset = np.linalg.lstsq(design, y[keep], rcond=None)[0]
        gain = float(np.clip(gain, 0.35, 2.5))
        offset = float(np.median(y[keep] - gain * x[keep]))
        residual = y - (gain * x + offset)
        center = float(np.median(residual[keep]))
        mad = float(1.4826 * np.median(np.abs(residual[keep] - center)))
        if mad <= 1e-8:
            break
        new_keep = np.abs(residual - center) <= 3.5 * mad
        if new_keep.sum() < 20 or np.array_equal(new_keep, keep):
            break
        keep = new_keep
    residual = y - (gain * x + offset)
    sigma = float(1.4826 * np.median(np.abs(residual - np.median(residual))))
    return gain, offset, float(np.clip(sigma, 10.0, 60.0))

def initial_rate(md, z, tvt, window=30):
    md, z, tvt = md[-window:], z[-window:], tvt[-window:]
    delta_md = np.diff(md)
    valid = delta_md > 0
    if valid.sum() < 3:
        return 0.0
    rates = np.diff(tvt + z)[valid] / delta_md[valid]
    return float(np.clip(np.median(rates), -0.25, 0.25))

def systematic_resample(weights, rng):
    cumulative = np.cumsum(weights)
    cumulative[-1] = 1.0
    points = rng.uniform(0.0, 1.0 / len(weights)) + np.arange(len(weights)) / len(weights)
    return np.searchsorted(cumulative, points, side='left')

In [ ]:
def run_seed(seed, md, z, gr, previous_md, anchor_u, rate0, tw_tvt, expected_hgr, gr_sigma):
    rng = np.random.default_rng(seed)
    u = anchor_u + 4.5 * rng.standard_normal(N_PARTICLES)
    rate = rate0 + 0.01 * rng.standard_normal(N_PARTICLES)
    weights = np.full(N_PARTICLES, 1.0 / N_PARTICLES)
    prediction = np.empty(len(md))
    log_evidence = 0.0
    for row in range(len(md)):
        delta_md = max(float(md[row] - previous_md), 1.0)
        rate = 0.998 * rate + 0.002 * rng.standard_normal(N_PARTICLES)
        u = u + rate * delta_md + 0.005 * rng.standard_normal(N_PARTICLES)
        particle_tvt = np.clip(u - z[row], tw_tvt[0] - 100.0, tw_tvt[-1] + 100.0)
        u = particle_tvt + z[row]
        if np.isfinite(gr[row]):
            expected = np.interp(particle_tvt, tw_tvt, expected_hgr)
            scaled = (gr[row] - expected) / gr_sigma
            likelihood = np.exp(np.clip(-0.5 * scaled * scaled, -50.0, 0.0))
            evidence = float(weights @ likelihood)
            log_evidence += np.log(max(evidence, 1e-300))
            weights *= likelihood
            total = float(weights.sum())
            weights = weights / total if total > 1e-300 else np.full(N_PARTICLES, 1.0 / N_PARTICLES)
        effective = 1.0 / float(weights @ weights)
        if effective < 0.5 * N_PARTICLES:
            selected = systematic_resample(weights, rng)
            u = u[selected] + 0.10 * rng.standard_normal(N_PARTICLES)
            rate = rate[selected] + 0.001 * rng.standard_normal(N_PARTICLES)
            weights.fill(1.0 / N_PARTICLES)
        prediction[row] = float(weights @ (u - z[row]))
        previous_md = md[row]
    return prediction, log_evidence

def particle_ensemble(frame, typewell, seed_base):
    boundary = split_boundary(frame)
    md = frame['MD'].to_numpy(dtype=float)
    z = frame['Z'].to_numpy(dtype=float)
    gr = frame['GR'].to_numpy(dtype=float)
    known = frame['TVT_input'].to_numpy(dtype=float)[:boundary]
    tw = typewell[['TVT', 'GR']].dropna().sort_values('TVT').drop_duplicates('TVT')
    tw_tvt = tw['TVT'].to_numpy(dtype=float)
    tw_gr = tw['GR'].to_numpy(dtype=float)
    gain, offset, gr_sigma = calibrate_gr(np.interp(known, tw_tvt, tw_gr), gr[:boundary])
    expected_hgr = gain * tw_gr + offset
    rate0 = initial_rate(md[:boundary], z[:boundary], known)
    anchor_u = float(known[-1] + z[boundary - 1])
    paths, evidence = [], []
    for seed in range(seed_base, seed_base + N_SEEDS):
        prediction, log_ev = run_seed(
            seed, md[boundary:], z[boundary:], gr[boundary:],
            float(md[boundary - 1]), anchor_u, rate0,
            tw_tvt, expected_hgr, gr_sigma,
        )
        paths.append(prediction)
        evidence.append(log_ev)
    evidence = np.asarray(evidence, dtype=float)
    logits = (evidence - evidence.max()) / SEED_TEMPERATURE
    seed_weights = np.exp(np.clip(logits, -50.0, 0.0))
    seed_weights /= seed_weights.sum()
    return boundary, np.average(np.stack(paths), axis=0, weights=seed_weights)

## Deterministic true-start comparison

The sample is selected once from sorted well paths. For each well and each seed base, the tracker sees the same visible prefix and predicts exactly the supplied missing suffix. We retain row-level SSE for pooled RMSE and per-well RMSE for replication diagnostics.

In [ ]:
records = []
started = time.perf_counter()
for number, path in enumerate(paths, start=1):
    well_id = path.name.removesuffix('__horizontal_well.csv')
    frame = pd.read_csv(path)
    typewell = pd.read_csv(path.with_name(f'{well_id}__typewell.csv'))
    boundary = split_boundary(frame)
    truth = frame['TVT'].to_numpy(dtype=float)[boundary:]
    anchor = np.full(len(truth), float(frame.loc[boundary - 1, 'TVT_input']))
    row = {'well_id': well_id, 'n_eval': len(truth)}
    anchor_error = anchor - truth
    row['sse_anchor'] = float(anchor_error @ anchor_error)
    row['rmse_anchor'] = math.sqrt(row['sse_anchor'] / len(truth))
    for label, base in BASES.items():
        candidate_boundary, particle = particle_ensemble(
            frame, typewell, base + number * 1000,
        )
        assert candidate_boundary == boundary
        blend = (1.0 - PARTICLE_WEIGHT) * anchor + PARTICLE_WEIGHT * particle
        error = blend - truth
        row[f'sse_{label}'] = float(error @ error)
        row[f'rmse_{label}'] = math.sqrt(row[f'sse_{label}'] / len(truth))
    records.append(row)
    print(
        f'{number:02d}/{len(paths)} {well_id} '
        f'anchor={row["rmse_anchor"]:.2f} '
        f'overlap={row["rmse_overlap_plus_one"]:.2f} '
        f'disjoint={row["rmse_disjoint"]:.2f}'
    )
results = pd.DataFrame(records)
print('Runtime seconds:', round(time.perf_counter() - started, 1))

In [ ]:
labels = ['anchor', 'original', 'overlap_plus_one', 'disjoint']
summary_rows = []
for label in labels:
    per_well = results[f'rmse_{label}']
    summary_rows.append({
        'run': label,
        'pooled_rmse': math.sqrt(results[f'sse_{label}'].sum() / results['n_eval'].sum()),
        'macro_rmse': per_well.mean(),
        'p95': per_well.quantile(0.95),
        'worst': per_well.max(),
    })
summary = pd.DataFrame(summary_rows).set_index('run')
display(summary.style.format('{:.3f}'))

correlations = pd.DataFrame({
    'comparison': ['original vs overlap +1', 'original vs disjoint'],
    'seed_intersection': [7, 0],
    'pearson_well_rmse': [
        results['rmse_original'].corr(results['rmse_overlap_plus_one']),
        results['rmse_original'].corr(results['rmse_disjoint']),
    ],
    'mean_absolute_well_delta': [
        (results['rmse_original'] - results['rmse_overlap_plus_one']).abs().mean(),
        (results['rmse_original'] - results['rmse_disjoint']).abs().mean(),
    ],
})
display(correlations.style.format({'pearson_well_rmse': '{:.3f}', 'mean_absolute_well_delta': '{:.3f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(results['rmse_original'], results['rmse_overlap_plus_one'], alpha=0.75)
axes[1].scatter(results['rmse_original'], results['rmse_disjoint'], alpha=0.75, color='#c45a32')
for ax, title in zip(axes, ['Seven of eight seeds overlap', 'No seeds overlap']):
    limit = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, limit], [0, limit], 'k--', linewidth=1)
    ax.set(xlabel='original per-well RMSE', ylabel='replication per-well RMSE', title=title)
plt.tight_layout()
plt.show()

## Full 773-well evidence from the same implementation

The compact sample above keeps this public notebook quick to rerun. A separate full local audit used exactly the same 300-particle, eight-seed, temperature-20 model at all 773 supplied boundaries:

| Fixed 62.5% particle blend | Pooled RMSE | Five-fold OOF learned-weight RMSE |
| --- | ---: | ---: |
| Original base 20260717 | 13.1119 | 13.1544 |
| Overlapping base 20260718 | 13.0549 | 13.0937 |
| Disjoint base 20270717 | 13.3505 | 13.3577 |
| Anchor | 15.9099 | n/a |

The disjoint result still supports the method, but it is less similar than the overlapping check. That difference is the point of the audit.

## Reusable checklist

1. Log the complete seed set for every stochastic ensemble, not only a base integer.
2. Assert zero intersection when claiming an independent replication.
3. Compare per-well errors and tail overlap, not only one pooled score.
4. Preserve one fixed deployment rule while varying seeds; otherwise the audit confounds randomness with retuning.
5. Treat stochastic worst-well failures as a model problem. A nearby blend-weight leaderboard search will not solve wrong GR branches.

Different labels are metadata. Independence is a property of the random streams actually consumed.